# Grouped-Query Attention (GQA) — Exercise

Companion to [Attention Part 2 § Grouped-Query Attention](https://tanulsingh.github.io/blog/attention-architectures#grouped-query-attention-ainslie-et-al-2023).

GQA generalizes the spectrum:
- `n_kv_heads = n_heads` → standard MHA (every Q has its own K,V)
- `n_kv_heads = 1` → MQA (one shared K,V for all Q heads)
- `1 < n_kv_heads < n_heads` → GQA (groups of Q heads share K,V)

E.g., Llama 2 70B has `n_heads=64, n_kv_heads=8` → groups of 8 Q heads each share one K and one V.

The key implementation trick: after projecting K, V to shape `(batch, seq, n_kv_heads, d_head)`, **repeat them along the head axis** so each group of Q heads gets a copy of its shared K, V. Then standard scaled dot-product attention works.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn, torch.nn.functional as F, math

## TODO 1 — `GroupedQueryAttention`

Generalize the MQA pattern to support any `n_kv_heads` between 1 and `n_heads`. Constraint: `n_heads % n_kv_heads == 0` (groups must be equal-sized).

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, causal: bool = False):
        super().__init__()
        # TODO 1a: assert n_heads % n_kv_heads == 0
        # TODO 1b: store d_model, n_heads, n_kv_heads, d_head, causal
        # TODO 1c: W_q: nn.Linear(d_model, n_heads * d_head)         # = d_model
        # TODO 1d: W_k: nn.Linear(d_model, n_kv_heads * d_head)      # smaller!
        # TODO 1e: W_v: nn.Linear(d_model, n_kv_heads * d_head)      # smaller!
        # TODO 1f: W_o: nn.Linear(d_model, d_model)
        pass

    def forward(self, x):
        # x: (batch, seq, d_model)
        # TODO 1g: project to Q, K, V
        # TODO 1h: reshape Q to (batch, n_heads,    seq, d_head)
        # TODO 1i: reshape K, V to (batch, n_kv_heads, seq, d_head)
        # TODO 1j: repeat K, V along head axis to match n_heads:
        #   K_repeated = K.repeat_interleave(n_heads // n_kv_heads, dim=1)
        # TODO 1k: now K_repeated, V_repeated have shape (batch, n_heads, seq, d_head)
        # TODO 1l: standard scaled dot-product attention with optional causal mask
        # TODO 1m: reshape back and apply W_o
        pass

In [ ]:
# Sanity check:
#   - gqa = GroupedQueryAttention(d_model=64, n_heads=8, n_kv_heads=2)
#   - out = gqa(torch.randn(2, 10, 64))
#   - out.shape == (2, 10, 64)
#   - Verify: gqa(d=64, n=8, n_kv=8) is equivalent to MHA in param count
#   - Verify: gqa(d=64, n=8, n_kv=1) is equivalent to MQA in param count

## TODO 2 — Sweep cache sizes

For `d_model=512, n_heads=8`, compute the W_k parameter count for `n_kv_heads in [1, 2, 4, 8]`. Print as a table.

Expected: `n_kv_heads=1` (MQA) is 8× smaller than `n_kv_heads=8` (MHA), with the in-between values scaling linearly.

In [ ]:
# TODO 2:
#   - For each n_kv_heads in [1, 2, 4, 8]:
#       gqa = GroupedQueryAttention(512, 8, n_kv_heads)
#       Print: n_kv_heads, W_k param count, ratio vs MHA

## Run the tests

In [ ]:
from tests import run_gqa_tests
# run_gqa_tests(GroupedQueryAttention)